In [ ]:
#!/usr/bin/env python3
"""
Extract sigma(gamma p -> Upsilon(nS) p) versus W_gamma_p from a flattened UpsilonTree.

This assumes the tree contains event weights normalized to the ep luminosity used below.
The output cross section is obtained by dividing the corrected ep yield by the integrated
Weizsaecker-Williams photon flux in each W bin.
"""

import argparse
import math
import os
import sys

import ROOT
import numpy as np
import matplotlib.pyplot as plt
import mplhep as hep

hep.style.use(hep.style.ROOT)

ALPHA_EM = 1.0 / 137.035999084
M_E = 0.00051099895
M_P = 0.9382720813

UPSILON_STATES = {
    "1S": {"mass": 9.46030, "sigma": 0.053, "br_ee": 0.0234, "color": "royalblue"},
    "2S": {"mass": 10.02326, "sigma": 0.056, "br_ee": 0.0191, "color": "crimson"},
    "3S": {"mass": 10.35520, "sigma": 0.059, "br_ee": 0.0218, "color": "forestgreen"},
}


def photon_flux_dW(W, e_e, e_p, q2_max):
    """Equivalent photon flux dN_gamma/dW for ep photoproduction."""
    s_minus_mp2 = 4.0 * e_e * e_p
    y = (W * W - M_P * M_P) / s_minus_mp2
    if y <= 0.0 or y >= 1.0:
        return 0.0

    q2_min = (M_E * M_E * y * y) / (1.0 - y)
    if q2_min <= 0.0 or q2_min >= q2_max:
        return 0.0

    log_term = math.log(q2_max / q2_min)
    dndy = (ALPHA_EM / (2.0 * math.pi)) * (
        ((1.0 + (1.0 - y) ** 2) / y) * log_term
        - (2.0 * (1.0 - y) / y) * (1.0 - q2_min / q2_max)
    )
    dydw = 2.0 * W / s_minus_mp2
    return dndy * dydw


def integrated_flux(w_low, w_high, e_e, e_p, q2_max, n_steps=200):
    grid = np.linspace(w_low, w_high, n_steps)
    vals = np.array([photon_flux_dW(w, e_e, e_p, q2_max) for w in grid])
    return np.trapz(vals, grid)


def hist_weighted(values, weights, bins):
    y, _ = np.histogram(values, bins=bins, weights=weights)
    e2, _ = np.histogram(values, bins=bins, weights=weights * weights)
    return y.astype(float), np.sqrt(e2.astype(float))


def main():
    parser = argparse.ArgumentParser()
    # Made input optional to prevent notebook crashes; replace default with your actual file name
    parser.add_argument("input", nargs='?', default="~/merged_elecs_test_10t100_10x100.root", help="Input ROOT file containing UpsilonTree")
    parser.add_argument("--tree", default="UpsilonTree")
    parser.add_argument("--out", default="~/campaigns/Upsilon_Family_GammaP_corrected.png")
    parser.add_argument("--lumi-invfb", type=float, default=10.0)
    parser.add_argument("--electron-energy", type=float, default=10.0)
    parser.add_argument("--proton-energy", type=float, default=100.0)
    parser.add_argument("--q2-max", type=float, default=0.001)
    parser.add_argument("--w-min", type=float, default=10.0)
    parser.add_argument("--w-max", type=float, default=50.0)
    parser.add_argument("--n-w-bins", type=int, default=14)
    parser.add_argument(
        "--weight-convention",
        choices=["visible_decay", "inclusive_upsilon"],
        default="visible_decay",
        help="Use visible_decay if the tuple weight includes Upsilon -> ee branching. Use inclusive_upsilon otherwise.",
    )
    parser.add_argument("--sigma-epz-center", type=float, default=None)
    parser.add_argument("--sigma-epz-halfwidth", type=float, default=None)
    
    # Check if running in a jupyter notebook
    if 'ipykernel' in sys.modules:
        args = parser.parse_args(args=[])
    else:
        args = parser.parse_args()

    filename = os.path.expanduser(args.input)
    
    # Ensure output directory exists
    out_path = os.path.expanduser(args.out)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    
    df = ROOT.RDataFrame(args.tree, filename)

    w_bins = np.linspace(args.w_min, args.w_max, args.n_w_bins + 1)
    w_centers = 0.5 * (w_bins[:-1] + w_bins[1:])

    fig, ax = plt.subplots(figsize=(10, 8))

    hera_w = np.array([95.0, 175.0])
    hera_w_err = np.array([35.0, 45.0])
    hera_sig = np.array([160.0, 321.0])
    hera_err = np.array([70.0, 144.0])
    ax.errorbar(
        hera_w,
        hera_sig,
        xerr=hera_w_err,
        yerr=hera_err,
        fmt="^",
        color="gray",
        label=r"HERA $\gamma p \rightarrow \Upsilon(1S)p$",
        markersize=9,
        linewidth=2,
    )

    sigma_epz_center = args.sigma_epz_center
    sigma_epz_halfwidth = args.sigma_epz_halfwidth
    if sigma_epz_center is None:
        sigma_epz_center = 2.0 * args.electron_energy
    if sigma_epz_halfwidth is None:
        sigma_epz_halfwidth = 0.2 * args.electron_energy

    for name, props in UPSILON_STATES.items():
        mass_window = 3.0 * props["sigma"]
        mass_min = props["mass"] - mass_window
        mass_max = props["mass"] + mass_window

        reco_filter = (
            f"reco_mass >= {mass_min} && reco_mass <= {mass_max}"
            f" && abs(sigma_epz - {sigma_epz_center}) <= {sigma_epz_halfwidth}"
            f" && reco_W >= {args.w_min} && reco_W <= {args.w_max}"
        )
        true_filter = (
            f"true_mass >= {props['mass'] - 0.2} && true_mass <= {props['mass'] + 0.2}"
            f" && true_W >= {args.w_min} && true_W <= {args.w_max}"
        )

        reco = df.Filter(reco_filter).AsNumpy(["reco_W", "weight"])
        truth = df.Filter(true_filter).AsNumpy(["true_W", "weight"])

        if len(reco["reco_W"]) == 0 or len(truth["true_W"]) == 0:
            print(f"Upsilon({name}) skipped due to zero reco or truth entries")
            continue

        n_reco, err_reco = hist_weighted(reco["reco_W"], reco["weight"], w_bins)
        n_true, _ = hist_weighted(truth["true_W"], truth["weight"], w_bins)

        eff = np.divide(n_reco, n_true, out=np.zeros_like(n_reco), where=n_true > 0.0)

        sigma = np.zeros_like(n_reco)
        err_sigma = np.zeros_like(n_reco)
        for i, w_center in enumerate(w_centers):
            flux_int = integrated_flux(
                w_bins[i], w_bins[i + 1], args.electron_energy, args.proton_energy, args.q2_max
            )
            if n_reco[i] <= 0.0 or eff[i] <= 0.0 or flux_int <= 0.0:
                continue

            # L[fb^-1] times 1000 gives L[pb^-1].
            denom = args.lumi_invfb * 1000.0 * eff[i] * flux_int
            br_corr = 1.0
            if args.weight_convention == "visible_decay":
                br_corr = 1.0 / props["br_ee"]

            sigma[i] = (n_reco[i] / denom) * br_corr
            err_sigma[i] = (err_reco[i] / denom) * br_corr

        mask = sigma > 0.0
        ax.errorbar(
            w_centers[mask],
            sigma[mask],
            yerr=err_sigma[mask],
            fmt="o",
            color=props["color"],
            label=rf"ePIC projected $\Upsilon({name})$",
            markersize=8,
            linewidth=2,
        )

    sqrts = math.sqrt(M_P * M_P + 4.0 * args.electron_energy * args.proton_energy)
    ax.set_xlabel(r"$W_{\gamma p}$ [GeV]", fontsize=14)
    ax.set_ylabel(r"$\sigma(\gamma p \rightarrow \Upsilon(nS)p)$ [pb]", fontsize=14)
    ax.set_yscale("log")
    ax.set_xlim(args.w_min, max(args.w_max, 225.0))
    ax.set_ylim(1.0, 2000.0)
    ax.text(0.04, 0.94, "ePIC Performance", transform=ax.transAxes, fontsize=12, fontweight="bold", ha="left", va="top")
    ax.text(0.04, 0.88, rf"$e+p$, {args.electron_energy:.0f}$\times${args.proton_energy:.0f} GeV", transform=ax.transAxes, fontsize=11, ha="left", va="top")
    ax.text(0.04, 0.82, rf"$\sqrt{{s}} \simeq {sqrts:.0f}$ GeV, $Q^2 < {args.q2_max:g}$ GeV$^2$", transform=ax.transAxes, fontsize=11, ha="left", va="top")
    ax.text(0.04, 0.76, rf"$\mathcal{{L}}_{{ep}} = {args.lumi_invfb:g}$ fb$^{{-1}}$", transform=ax.transAxes, fontsize=11, ha="left", va="top")
    ax.legend(loc="lower right", fontsize=11)
    fig.tight_layout()
    fig.savefig(out_path, dpi=300)
    print(f"Saved {out_path}")


if __name__ == "__main__":
    main()